# Continuous predictor analysis | Issue #17

**Method v1.0** · Source: `data/allstate_claims_data.csv` · Unit: one claim · Reporting target: raw `loss` (USD).

Run all cells from a fresh kernel. This notebook analyzes all 14 `cont*` fields, generates reviewable evidence, and does **not** remove features. Correlations and bin trends are marginal associations, not causal effects or feature-importance scores.

## 1. Source and settings

The source must match the approved size, shape, and header. If the team has an approved SHA-256, enter it below; the computed hash is always recorded for independent comparison.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import tempfile
import uuid

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

METHOD_VERSION = "1.0.0"
EXPECTED_SIZE_BYTES = 70_025_339
EXPECTED_SHAPE = (188_318, 132)
EXPECTED_SHA256 = None  # Set only if the team has an approved reference hash.
DEFAULT_QUANTILE_BINS = 10
HISTOGRAM_BINS = 30
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
EXPECTED_HEADER = ["id"] + [f"cat{i}" for i in range(1, 117)] + CONT_COLS + ["loss"]


def locate_root():
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (folder / "data" / "allstate_claims_data.csv").is_file():
            return folder
    raise FileNotFoundError("Could not find data/allstate_claims_data.csv. Open the notebook within the repository.")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = locate_root()
SOURCE = ROOT / "data" / "allstate_claims_data.csv"
EVIDENCE_DIR = ROOT / "notebooks" / "final-deliverables" / "September" / "Gate 2" / "continuous_analysis_evidence"

source_size = SOURCE.stat().st_size
source_hash = sha256(SOURCE)
if source_size != EXPECTED_SIZE_BYTES:
    raise RuntimeError(f"Incorrect source size: {source_size:,} bytes; expected {EXPECTED_SIZE_BYTES:,}. Stop and investigate.")
if EXPECTED_SHA256 is not None and source_hash.lower() != EXPECTED_SHA256.lower():
    raise RuntimeError("Source SHA-256 mismatch. Stop and investigate.")

# Load strings as labels when needed; continuous fields must remain numeric.
df = pd.read_csv(SOURCE)
if df.shape != EXPECTED_SHAPE or df.columns.tolist() != EXPECTED_HEADER:
    raise RuntimeError(f"Source shape/header mismatch: {df.shape}. Stop and investigate.")

print("Source identity: PASS")
print(f"File size: {source_size:,} bytes | Shape: {df.shape}")
print(f"SHA-256: {source_hash}")

Source identity: PASS
File size: 70,025,339 bytes | Shape: (188318, 132)
SHA-256: 74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03


## 2. Check analysis inputs

Confirm the continuous columns are numeric, finite, and on the delivered 0–1 scale; `loss` must be numeric, finite, and positive. No source values are altered.

In [2]:
issues = []
for col in CONT_COLS + ["loss"]:
    s = df[col]
    if not pd.api.types.is_numeric_dtype(s):
        issues.append(f"{col}: not numeric ({s.dtype})")
        continue
    if s.isna().any() or not np.isfinite(s.to_numpy(dtype=float)).all():
        issues.append(f"{col}: missing or non-finite value(s)")
    if col in CONT_COLS and not s.between(0, 1).all():
        issues.append(f"{col}: value(s) outside [0, 1]")
    if col == "loss" and not s.gt(0).all():
        issues.append("loss: non-positive value(s)")

if issues:
    raise RuntimeError("Input validation failed; nothing has been published:\n" + "\n".join(issues))

# Derived only in memory. Original loss remains the reporting/evaluation scale.
log1p_loss = np.log1p(df["loss"])
if not np.isfinite(log1p_loss.to_numpy()).all():
    raise RuntimeError("Derived log1p_loss contains non-finite values.")
print("Continuous fields and loss: PASS")

Continuous fields and loss: PASS


## 3. Continuous-field inventory

One row per predictor. The IQR here describes the **predictor distribution**, not the loss IQR reported later for quantile bins.

In [3]:
inventory = df[CONT_COLS].describe(percentiles=[0.25, 0.5, 0.75]).T
inventory.index.name = "field"
inventory = inventory.rename(columns={"25%": "q1", "50%": "median", "75%": "q3"})
inventory["iqr"] = inventory["q3"] - inventory["q1"]
inventory["unique_count"] = df[CONT_COLS].nunique(dropna=False)
inventory["missing_count"] = df[CONT_COLS].isna().sum()
inventory["missing_pct"] = df[CONT_COLS].isna().mean() * 100
inventory = inventory.reset_index()[["field", "count", "mean", "std", "min", "q1", "median", "q3", "max", "iqr", "unique_count", "missing_count", "missing_pct"]]
display(inventory)

,field,count,mean,std,min,q1,median,q3,max,iqr,unique_count,missing_count,missing_pct
0,cont1,188318.0,0.493861,0.187640,0.000016,0.346090,0.475784,0.623912,0.984975,0.277822,647,0,0.0
1,cont2,188318.0,0.507188,0.207202,0.001149,0.358319,0.555782,0.681761,0.862654,0.323442,33,0,0.0
2,cont3,188318.0,0.498918,0.202105,0.002634,0.336963,0.527991,0.634224,0.944251,0.297261,76,0,0.0
3,cont4,188318.0,0.491812,0.211292,0.176921,0.327354,0.452887,0.652072,0.954297,0.324718,112,0,0.0
4,cont5,188318.0,0.487428,0.209027,0.281143,0.281143,0.422268,0.643315,0.983674,0.362172,141,0,0.0
5,cont6,188318.0,0.490945,0.205273,0.012683,0.336105,0.440945,0.655021,0.997162,0.318916,2573,0,0.0
6,cont7,188318.0,0.484970,0.178450,0.069503,0.350175,0.438285,0.591045,1.000000,0.240870,5632,0,0.0
7,cont8,188318.0,0.486437,0.199370,0.236880,0.312800,0.441060,0.623580,0.980200,0.310780,201,0,0.0
8,cont9,188318.0,0.485506,0.181660,0.000080,0.358970,0.441450,0.566820,0.995400,0.207850,347,0,0.0
9,cont10,188318.0,0.498066,0.185877,0.000000,0.364580,0.461190,0.614590,0.994980,0.250010,174,0,0.0


## 4. Correlations with the target

For every predictor, calculate Pearson and Spearman correlations with both raw `loss` and `log1p_loss`. Do not interpret any coefficient as feature importance or causation.

In [4]:
corr_rows = []
for col in CONT_COLS:
    corr_rows.append({
        "field": col,
        "pearson_loss": df[col].corr(df["loss"], method="pearson"),
        "spearman_loss": df[col].corr(df["loss"], method="spearman"),
        "pearson_log1p_loss": df[col].corr(log1p_loss, method="pearson"),
        "spearman_log1p_loss": df[col].corr(log1p_loss, method="spearman"),
    })
target_correlations = pd.DataFrame(corr_rows)
display(target_correlations)

,field,pearson_loss,spearman_loss,pearson_log1p_loss,spearman_log1p_loss
0,cont1,-0.010237,-0.017641,-0.007335,-0.017641
1,cont2,0.141528,0.080066,0.104666,0.080066
2,cont3,0.111053,0.068353,0.081548,0.068353
3,cont4,-0.035831,-0.027878,-0.027523,-0.027878
4,cont5,-0.011355,-0.015114,-0.014958,-0.015114
5,cont6,0.040967,0.019697,0.031517,0.019697
6,cont7,0.119799,0.054928,0.085095,0.054928
7,cont8,0.030508,0.027495,0.032042,0.027495
8,cont9,0.014456,0.004401,0.017417,0.004401
9,cont10,0.020236,0.003042,0.010604,0.003042


## 5. Predictor-predictor correlations

The heatmap uses Pearson correlation. The pairs table contains all 91 distinct pairs; the first 10 are shown for review. Correlation alone does not justify dropping a predictor in September.

In [5]:
predictor_corr = df[CONT_COLS].corr(method="pearson")
pair_rows = []
for i, a in enumerate(CONT_COLS):
    for b in CONT_COLS[i + 1:]:
        r = float(predictor_corr.loc[a, b])
        pair_rows.append({"field_1": a, "field_2": b, "pearson_r": r, "absolute_r": abs(r)})
predictor_pairs = pd.DataFrame(pair_rows).sort_values("absolute_r", ascending=False).reset_index(drop=True)
display(predictor_pairs.head(10))

,field_1,field_2,pearson_r,absolute_r
0,cont11,cont12,0.994384,0.994384
1,cont1,cont9,0.929912,0.929912
2,cont6,cont10,0.883351,0.883351
3,cont6,cont13,0.815091,0.815091
4,cont1,cont10,0.808551,0.808551
5,cont6,cont9,0.797544,0.797544
6,cont9,cont10,0.785697,0.785697
7,cont6,cont12,0.785144,0.785144
8,cont6,cont11,0.773745,0.773745
9,cont1,cont6,0.758315,0.758315


## 6. Quantile-bin rule

Default: 10 requested quantile bins per predictor (`pd.qcut`, right-closed, lowest value included). Repeated values can collapse boundaries; we keep the actual distinct bins and **never split equal values artificially**. Labels `B1`, `B2`, ... increase with predictor value. A constant field receives one bin. Record the exact edges and support for every observed bin.

In [6]:
bin_inventory_rows = []
bin_rows = []
bin_assignments = {}  # Derived support fields; not written into the source CSV.

for col in CONT_COLS:
    series = df[col]
    if series.nunique(dropna=False) == 1:
        # qcut cannot form a positive-width interval from a constant feature.
        edges = np.array([float(series.iloc[0]), float(series.iloc[0])])
        codes = pd.Series(np.zeros(len(df), dtype=np.int64), index=df.index)
        method_note = "constant field: one bin"
    else:
        raw_codes, edges = pd.qcut(
            series, q=DEFAULT_QUANTILE_BINS, labels=False,
            retbins=True, duplicates="drop"
        )
        codes = pd.Series(raw_codes, index=df.index)
        method_note = "qcut; duplicate quantile edges dropped"

    actual_bins = int(len(edges) - 1)
    if actual_bins < 1 or codes.isna().any():
        raise RuntimeError(f"{col}: could not assign every observation to a valid bin.")
    codes = codes.astype("int64")
    observed_codes = sorted(codes.unique().tolist())
    if observed_codes != list(range(actual_bins)):
        raise RuntimeError(f"{col}: empty/nonconsecutive bin codes; investigate binning.")

    field_name = f"{col}_quantile_bin"
    bin_assignments[field_name] = codes + 1  # integer labels 1 through actual_bins
    bin_inventory_rows.append({
        "field": col, "derived_field": field_name,
        "requested_bins": DEFAULT_QUANTILE_BINS, "actual_bins": actual_bins,
        "unique_values": int(series.nunique()), "bin_edges_json": json.dumps([float(v) for v in edges]),
        "method_note": method_note,
    })

    working = pd.DataFrame({"bin_number": codes + 1, "value": series, "loss": df["loss"]})
    for bin_number, group in working.groupby("bin_number", sort=True):
        index = int(bin_number) - 1
        q1, q3 = group["loss"].quantile([0.25, 0.75]).tolist()
        bin_rows.append({
            "field": col, "derived_field": field_name,
            "bin_number": int(bin_number), "bin_label": f"B{bin_number}",
            "left_edge": float(edges[index]), "right_edge": float(edges[index + 1]),
            "observed_min": float(group["value"].min()),
            "observed_max": float(group["value"].max()),
            "support": int(len(group)), "mean_loss": float(group["loss"].mean()),
            "median_loss": float(group["loss"].median()),
            "loss_q1": float(q1), "loss_q3": float(q3), "loss_iqr": float(q3 - q1),
        })

bin_inventory = pd.DataFrame(bin_inventory_rows)
bin_summary = pd.DataFrame(bin_rows)
display(bin_inventory[["field", "requested_bins", "actual_bins", "unique_values", "method_note"]])
display(bin_summary.head(15))

,field,requested_bins,actual_bins,unique_values,method_note
0,cont1,10,10,647,qcut; duplicate quantile edges dropped
1,cont2,10,9,33,qcut; duplicate quantile edges dropped
2,cont3,10,10,76,qcut; duplicate quantile edges dropped
3,cont4,10,10,112,qcut; duplicate quantile edges dropped
4,cont5,10,8,141,qcut; duplicate quantile edges dropped
5,cont6,10,10,2573,qcut; duplicate quantile edges dropped
6,cont7,10,10,5632,qcut; duplicate quantile edges dropped
7,cont8,10,10,201,qcut; duplicate quantile edges dropped
8,cont9,10,10,347,qcut; duplicate quantile edges dropped
9,cont10,10,10,174,qcut; duplicate quantile edges dropped


,field,derived_field,bin_number,bin_label,left_edge,right_edge,observed_min,observed_max,support,mean_loss,median_loss,loss_q1,loss_q3,loss_iqr
0,cont1,cont1_quantile_bin,1,B1,0.000016,0.282512,0.000016,0.282512,19251,3036.539403,2209.820,1228.3150,4013.9150,2785.6000
1,cont1,cont1_quantile_bin,2,B2,0.282512,0.325401,0.283689,0.325401,20166,3045.865114,2194.015,1214.1350,4019.0625,2804.9275
2,cont1,cont1_quantile_bin,3,B3,0.325401,0.372785,0.326675,0.372785,17328,3215.119048,2192.145,1208.5625,4175.8100,2967.2475
3,cont1,cont1_quantile_bin,4,B4,0.372785,0.452710,0.374142,0.452710,19430,3133.854527,2136.540,1216.3150,3943.3175,2727.0025
4,cont1,cont1_quantile_bin,5,B5,0.452710,0.475784,0.454147,0.475784,19216,3006.873916,2045.835,1147.8550,3802.7625,2654.9075
5,cont1,cont1_quantile_bin,6,B6,0.475784,0.501862,0.477231,0.501862,17969,3046.193525,2111.940,1200.0700,3926.4400,2726.3700
6,cont1,cont1_quantile_bin,7,B7,0.501862,0.555279,0.503312,0.555279,18512,3064.813792,2110.445,1175.8525,3924.5725,2748.7200
7,cont1,cont1_quantile_bin,8,B8,0.555279,0.642763,0.556710,0.642763,19980,2777.374131,1937.035,1133.5525,3410.5400,2276.9875
8,cont1,cont1_quantile_bin,9,B9,0.642763,0.772995,0.644094,0.772995,17941,2938.553535,2014.890,1184.6600,3648.2900,2463.6300
9,cont1,cont1_quantile_bin,10,B10,0.772995,0.984975,0.774011,0.984975,18525,3132.963475,2237.600,1352.8200,3799.6500,2446.8300


## 7. Validate calculated evidence

All 14 fields, 56 target coefficients, 91 predictor pairs, and complete bin coverage are required. Stop before saving if any check fails.

In [7]:
assert len(inventory) == len(target_correlations) == len(bin_inventory) == 14
assert target_correlations.drop(columns="field").shape == (14, 4)
assert target_correlations.drop(columns="field").notna().all().all(), "Undefined target correlation; review constant fields."
assert len(predictor_pairs) == 91 and predictor_pairs["pearson_r"].notna().all()
assert predictor_corr.shape == (14, 14)
assert bin_inventory["actual_bins"].between(1, DEFAULT_QUANTILE_BINS).all()
assert bin_inventory["actual_bins"].sum() == len(bin_summary)
assert (bin_summary.groupby("field")["support"].sum().reindex(CONT_COLS) == len(df)).all()
assert all(s.notna().all() and len(s) == len(df) for s in bin_assignments.values())
assert bin_summary["support"].gt(0).all()
assert (bin_summary["loss_iqr"] >= 0).all()
assert bin_summary["bin_number"].ge(1).all()
print("Numerical and bin-coverage checks: PASS")

Numerical and bin-coverage checks: PASS


## 8. Generate plots and publish evidence

Each field gets a histogram **and** boxplot, plus a support/mean/median bin plot. One annotated Pearson heatmap completes the figures. The CSVs and figures are generated into a temporary folder and published together **only after all plots succeed**. Never hand-edit the generated results.

In [8]:
EVIDENCE_DIR.parent.mkdir(parents=True, exist_ok=True)
staging = Path(tempfile.mkdtemp(prefix="continuous_analysis_staging_", dir=EVIDENCE_DIR.parent))
figures_dir = staging / "figures"
figures_dir.mkdir()

try:
    inventory.to_csv(staging / "continuous_inventory.csv", index=False)
    target_correlations.to_csv(staging / "target_correlations.csv", index=False)
    predictor_corr.rename_axis("field").to_csv(staging / "predictor_pearson_matrix.csv")
    predictor_pairs.to_csv(staging / "predictor_pairs.csv", index=False)
    bin_inventory.to_csv(staging / "bin_inventory.csv", index=False)
    bin_summary.to_csv(staging / "bin_summary.csv", index=False)

    # 14 distribution figure files; each includes both required views.
    for col in CONT_COLS:
        fig, (ax_hist, ax_box) = plt.subplots(1, 2, figsize=(11, 3.5))
        ax_hist.hist(df[col], bins=HISTOGRAM_BINS)
        ax_hist.set(title=f"{col}: histogram", xlabel="Delivered value (0–1)", ylabel="Claims", xlim=(0, 1))
        ax_box.boxplot(df[col].to_numpy(), vert=False)
        ax_box.set(title=f"{col}: boxplot", xlabel="Delivered value (0–1)", xlim=(0, 1))
        fig.tight_layout()
        fig.savefig(figures_dir / f"{col}_distribution.png", dpi=160)
        plt.close(fig)

    # 14 bin-pattern figure files; support on left axis, loss in USD on right.
    for col in CONT_COLS:
        part = bin_summary.loc[bin_summary["field"] == col].sort_values("bin_number")
        x = part["bin_number"].to_numpy(dtype=int)
        fig, ax_support = plt.subplots(figsize=(9, 4))
        bars = ax_support.bar(x, part["support"], alpha=0.35, label="Support")
        ax_support.set(xlabel=f"{col} bin (in increasing value order)", ylabel="Claims", xticks=x)
        ax_support.set_xticklabels(part["bin_label"].tolist())
        ax_loss = ax_support.twinx()
        line_mean, = ax_loss.plot(x, part["mean_loss"], marker="o", label="Mean loss")
        line_median, = ax_loss.plot(x, part["median_loss"], marker="s", label="Median loss")
        ax_loss.set_ylabel("Loss (USD)")
        ax_support.legend([bars, line_mean, line_median], ["Support", "Mean loss", "Median loss"], loc="best")
        ax_support.set_title(f"{col}: bin support and raw target pattern")
        fig.tight_layout()
        fig.savefig(figures_dir / f"{col}_bin_pattern.png", dpi=160)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 9))
    heat = ax.imshow(predictor_corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(14), CONT_COLS, rotation=90)
    ax.set_yticks(range(14), CONT_COLS)
    for i in range(14):
        for j in range(14):
            r = predictor_corr.iat[i, j]
            ax.text(j, i, f"{r:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(r) >= 0.6 else "black")
    fig.colorbar(heat, ax=ax, label="Pearson r")
    ax.set_title("Continuous predictor Pearson correlations")
    fig.tight_layout()
    fig.savefig(figures_dir / "predictor_pearson_heatmap.png", dpi=180)
    plt.close(fig)

    figure_count = len(list(figures_dir.glob("*.png")))
    assert figure_count == 29, f"Expected 29 figure files; found {figure_count}"

    # Companion definitions for the data dictionary. These are not claims that
    # quantile bin columns were delivered with the source data.
    registry = [{
        "field_name": "log1p_loss", "purpose": "Exploratory view of target distribution",
        "inputs": "loss", "exact_derivation": "numpy.log1p(loss) = ln(1 + loss)",
        "allowed_values": "finite values > 0 for this positive-loss source",
        "field_version": METHOD_VERSION, "owner": "Liam", "observed_dtype": str(log1p_loss.dtype),
        "observed_unique_count": int(log1p_loss.nunique()), "observed_missing_count": int(log1p_loss.isna().sum()),
        "notes": "Exploratory only; original loss remains the reporting and MAE scale."
    }]
    for record in bin_inventory_rows:
        field = record["field"]
        derived = record["derived_field"]
        registry.append({
            "field_name": derived, "purpose": "Quantile-bin support and target pattern analysis",
            "inputs": field,
            "exact_derivation": f"pd.qcut({field}, q={DEFAULT_QUANTILE_BINS}, labels=False, duplicates='drop'); label code + 1; constant field -> B1",
            "allowed_values": f"integer bin labels 1..{record['actual_bins']}",
            "field_version": METHOD_VERSION, "owner": "Liam",
            "observed_dtype": str(bin_assignments[derived].dtype),
            "observed_unique_count": int(bin_assignments[derived].nunique()),
            "observed_missing_count": int(bin_assignments[derived].isna().sum()),
            "notes": f"right-closed bins, lowest included; exact edges: {record['bin_edges_json']}"
        })
    pd.DataFrame(registry).to_csv(staging / "derived_field_registry.csv", index=False)

    metadata = {
        "method_version": METHOD_VERSION, "source_path": "data/allstate_claims_data.csv",
        "source_bytes": source_size, "source_sha256": source_hash,
        "source_rows": len(df), "continuous_fields": CONT_COLS,
        "target": "loss", "target_unit": "USD", "transformed_target": "log1p_loss",
        "histogram_bins": HISTOGRAM_BINS, "requested_quantile_bins": DEFAULT_QUANTILE_BINS,
        "quantile_method": "pd.qcut; duplicate edges dropped; right-closed, lowest included; constant field one bin",
        "correlation_methods": ["Pearson", "Spearman"],
        "feature_removal_in_september": "None based solely on target or predictor correlations",
        "figure_count": figure_count,
        "limitations": "Associations are marginal and observational; no causal or individual reserve inference.",
    }
    (staging / "method_and_source.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

    # Hash all saved evidence files, excluding the manifest itself.
    manifest_rows = []
    for path in sorted(staging.rglob("*")):
        if path.is_file():
            manifest_rows.append({"artifact": path.relative_to(staging).as_posix(), "bytes": path.stat().st_size, "sha256": sha256(path)})
    pd.DataFrame(manifest_rows).to_csv(staging / "artifact_manifest.csv", index=False)

    # Publish as a group. A failure before this point leaves prior evidence intact.
    previous = EVIDENCE_DIR.parent / f".{EVIDENCE_DIR.name}_previous_{uuid.uuid4().hex}"
    had_previous = EVIDENCE_DIR.exists()
    if had_previous:
        EVIDENCE_DIR.rename(previous)
    try:
        staging.rename(EVIDENCE_DIR)
    except Exception:
        if had_previous:
            previous.rename(EVIDENCE_DIR)
        raise
    if had_previous:
        shutil.rmtree(previous)
except Exception:
    if staging.exists():
        shutil.rmtree(staging)
    raise

print(f"EVIDENCE PUBLISHED: {EVIDENCE_DIR.relative_to(ROOT)}")
print(f"Figures: {figure_count} (14 histogram+boxplot files, 14 bin plots, 1 heatmap)")
print(f"Tables: inventory, target correlations, predictor matrix/pairs, bin inventory/summary")

EVIDENCE PUBLISHED: notebooks\final-deliverables\September\Gate 2\continuous_analysis_evidence
Figures: 29 (14 histogram+boxplot files, 14 bin plots, 1 heatmap)
Tables: inventory, target correlations, predictor matrix/pairs, bin inventory/summary


## 9. Recheck the published bundle

Verify that the exact saved files match the manifest. The independent reviewer should rerun the notebook from a clean kernel, compare the recorded source hash, and inspect the plots and bin counts. Record reviewer, disagreements, and reviewed artifact version in the team's Gate 2 review record.

In [9]:
manifest = pd.read_csv(EVIDENCE_DIR / "artifact_manifest.csv")
for entry in manifest.itertuples(index=False):
    path = EVIDENCE_DIR / entry.artifact
    assert path.is_file() and path.stat().st_size == entry.bytes and sha256(path) == entry.sha256, entry.artifact
assert len(manifest) == 37, f"Unexpected manifest count: {len(manifest)}"  # 29 figures + 7 CSV + 1 JSON
print("CONTINUOUS ANALYSIS: PASS")
print(f"Fields: {len(CONT_COLS)} | Target correlations: {14*4} | Predictor pairs: {len(predictor_pairs)}")
print(f"Bin coverage: {len(df):,} claims per field | Saved figures: {figure_count}")
print("Published bundle integrity: PASS | No features removed")

CONTINUOUS ANALYSIS: PASS
Fields: 14 | Target correlations: 56 | Predictor pairs: 91
Bin coverage: 188,318 claims per field | Saved figures: 29
Published bundle integrity: PASS | No features removed
